# 20 — SD Inpaint 변형 자연스러움 개선 (Phase 7-N)

**문제 진단**:
기존 노트북 13의 SD Inpaint 결과가 미적으로 자연스럽지 않다는 교수님 피드백.

**원인 분석**:
1. Prompt에 `"same person, same face"` 키워드 과도 강조 → identity 보존 우선 → **변형이 거의 안 일어남**
2. `strength` 파라미터 미설정 → 기본값 의존
3. Mask dilate가 작아 시술 영역이 좁음

**개선 방향 (3단계 실험)**:
1. **Prompt 강화** — "same person" 제거 + 시술 효과 키워드 강화
2. **Strength 튜닝** — 0.7 / 0.85 / 1.0 비교
3. **Guidance Scale 튜닝** — 7.0 / 8.5 / 10.0 비교

**최종 목표**: Identity 적정 유지 (LPIPS ≤ 0.15) + 시술 효과 강화 (사용자 만족도 ≥ 4.0/5.0)

## 1. 환경 셋업

In [ ]:
!nvidia-smi

In [ ]:
!pip install --quiet diffusers transformers accelerate safetensors albumentations mediapipe pyyaml

In [ ]:
import torch, time, os, sys
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
print('device:', device)

In [ ]:
# Git clone (project 모듈)
REPO_LOCAL = '/content/cv-final'
if not os.path.exists(REPO_LOCAL):
    !git clone https://github.com/keonU206/cv-final.git {REPO_LOCAL}
else:
    !cd {REPO_LOCAL} && git pull --quiet

for m in list(sys.modules.keys()):
    if m.startswith('project'):
        del sys.modules[m]

if REPO_LOCAL not in sys.path:
    sys.path.insert(0, REPO_LOCAL)

import importlib
importlib.invalidate_caches()

# Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/cv-final'
OUTPUT_DIR = Path(DRIVE_ROOT) / 'data' / 'outputs' / 'sd_enhanced'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('✅ 환경 준비 완료')
print(f'   OUTPUT_DIR: {OUTPUT_DIR}')

## 2. 샘플 이미지 + Mask 준비

In [ ]:
from project.input_generator import make_mask, load_procedures

SIZE = 512  # 256 → 512 업스케일 (해상도 ↑ for 더 자연스러운 결과)

# 샘플 이미지 로드 (Drive 또는 업로드)
from google.colab import files
print('📷 사진 업로드 (얼굴 정면 사진):')
uploaded = files.upload()
img_path = list(uploaded.keys())[0]

image_rgb = cv2.imread(img_path)
image_rgb = cv2.cvtColor(image_rgb, cv2.COLOR_BGR2RGB)
image_rgb = cv2.resize(image_rgb, (SIZE, SIZE))
print(f'이미지 크기: {image_rgb.shape}')

plt.figure(figsize=(6, 6))
plt.imshow(image_rgb)
plt.title('원본 사진 (Before)')
plt.axis('off')
plt.show()

In [ ]:
# 시술 영역 mask 생성 (코끝)
import mediapipe as mp

# MediaPipe 478 landmark 추출
BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

# 모델 파일 다운로드
!wget -q -O /tmp/face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='/tmp/face_landmarker.task'),
    running_mode=VisionRunningMode.IMAGE,
    num_faces=1,
)
landmarker = FaceLandmarker.create_from_options(options)

mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
result = landmarker.detect(mp_image)
landmarks_2d = np.array([[lm.x * SIZE, lm.y * SIZE] for lm in result.face_landmarks[0]])
print(f'478 landmark 추출 완료: {landmarks_2d.shape}')

In [ ]:
# 시술 영역 mask 생성
procedures_db = load_procedures()

PROC_ID = 'nose_tip'  # 또는 'double_eyelid', 'jaw_v_line'

# procedure dict 추출 (yaml에서 로드)
procedure = procedures_db[PROC_ID]

# Mask 생성 (실제 make_mask 시그니처: landmarks, procedure, size)
mask_raw = make_mask(
    landmarks=landmarks_2d.astype(np.int32),
    procedure=procedure,
    size=SIZE,
)

# (H, W, 1) → (H, W) 차원 정리
if mask_raw.ndim == 3:
    mask = mask_raw[:, :, 0]
else:
    mask = mask_raw

# 변형 영역 확장 (dilate 추가 — 시술 효과 강조)
MASK_DILATE_OVERRIDE = {
    'nose_tip': 18,
    'double_eyelid': 10,
    'jaw_v_line': 22,
}
dilate_size = MASK_DILATE_OVERRIDE[PROC_ID]
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dilate_size, dilate_size))
mask = cv2.dilate(mask, kernel, iterations=1)

print(f'Mask 영역: {(mask > 0).sum():,} 픽셀')
print(f'Dilate 추가: {dilate_size}px')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image_rgb); axes[0].set_title('원본'); axes[0].axis('off')
axes[1].imshow(mask, cmap='gray'); axes[1].set_title(f'Mask (dilate +{dilate_size}px)'); axes[1].axis('off')
overlay = image_rgb.copy()
overlay[mask > 0] = (overlay[mask > 0] * 0.5 + np.array([255, 0, 0]) * 0.5).astype(np.uint8)
axes[2].imshow(overlay); axes[2].set_title('Overlay'); axes[2].axis('off')
plt.tight_layout()
plt.show()

## 3. SD Inpaint 모델 로드

In [ ]:
from diffusers import StableDiffusionInpaintPipeline, DPMSolverMultistepScheduler

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    'runwayml/stable-diffusion-inpainting',
    torch_dtype=torch.float16,
    safety_checker=None,
).to(device)

# 더 빠른 sampler
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

print('✅ SD Inpaint 모델 로드 완료 (FP16)')

## 4. ⭐ Prompt 개선 (3가지 강도 비교)

기존 문제: `"same person, same face"` 과도 강조 → 변형 거의 안 됨

**3가지 강도로 비교**:
- **Weak** (기존): identity 강조, 변형 약함
- **Medium**: identity + 시술 효과 균형
- **Strong**: 시술 효과 강조, identity는 prompt 없이 SD 자체에 맡김

In [ ]:
# 3가지 강도의 prompt
PROMPTS_BY_STRENGTH = {
    'nose_tip': {
        'weak': {  # 기존 방식 (변형 약함)
            'positive': 'same person, same face, slightly refined nose tip, subtle natural change, photorealistic, Korean beauty',
            'negative': 'different person, different face, deformed, ugly, blurry, distorted, cartoon, anime',
        },
        'medium': {  # 균형 (추천)
            'positive': 'refined slim nose tip, elegant nose surgery result, natural Korean beauty, photorealistic portrait, smooth skin, high quality',
            'negative': 'deformed, ugly, blurry, distorted, cartoon, anime, oversaturated, asymmetric, bad anatomy, low quality',
        },
        'strong': {  # 강한 변형 (시술 효과 명확)
            'positive': 'beautifully refined slim nose tip after professional rhinoplasty, elegant aesthetic nose surgery, perfect Korean beauty standard, ultra photorealistic, smooth flawless skin, professional portrait photography, sharp focus, high resolution',
            'negative': 'wide nose, large nose tip, ugly, deformed, blurry, low quality, cartoon, anime, sketch, oversaturated, asymmetric, bad anatomy, watermark, text',
        },
    },
    'double_eyelid': {
        'weak': {
            'positive': 'same person, slightly visible double eyelid, subtle natural eyes, photorealistic',
            'negative': 'different person, deformed eyes, blurry, cartoon',
        },
        'medium': {
            'positive': 'natural double eyelid crease, bright open eyes, elegant Korean beauty, photorealistic portrait',
            'negative': 'closed eyes, asymmetric eyes, deformed, blurry, anime, low quality',
        },
        'strong': {
            'positive': 'beautiful natural double eyelid after blepharoplasty, bright clear large eyes, elegant Korean beauty standard, ultra photorealistic, perfect aesthetic eyes, professional portrait',
            'negative': 'closed eyes, monolid, asymmetric, deformed, ugly, blurry, cartoon, anime, low quality, heavy makeup',
        },
    },
    'jaw_v_line': {
        'weak': {
            'positive': 'same person, slightly slimmer jaw, subtle natural face shape, photorealistic',
            'negative': 'different person, wide jaw, deformed, blurry',
        },
        'medium': {
            'positive': 'slim v-line jaw, refined chin, elegant Korean face shape, photorealistic',
            'negative': 'wide jaw, square face, masculine jaw, deformed, blurry, cartoon',
        },
        'strong': {
            'positive': 'beautiful slim v-line jaw after professional surgery, elegantly refined sharp chin, perfect Korean beauty standard face shape, ultra photorealistic portrait, smooth flawless skin, professional photography',
            'negative': 'wide square jaw, masculine face, double chin, deformed, ugly, blurry, low quality, cartoon, anime, heavy makeup, oversaturated',
        },
    },
}

print(f'설정한 시술: {PROC_ID}')
for strength_name, prompts in PROMPTS_BY_STRENGTH[PROC_ID].items():
    print(f'\n[{strength_name.upper()}]')
    print(f'  positive: "{prompts["positive"][:80]}..."')
    print(f'  negative: "{prompts["negative"][:60]}..."')

## 5. ⭐ 실험 — 3 강도 × 3 guidance × strength 비교

총 9개 결과 생성 (Prompt 3개 × Guidance 3개)

In [ ]:
# 실험 설정
INFERENCE_STEPS = 50
GUIDANCE_SCALES = [7.0, 8.5, 10.0]  # 낮음 / 중간 / 높음
PROMPT_STRENGTHS = ['weak', 'medium', 'strong']

# PIL 변환
image_pil = Image.fromarray(image_rgb)
mask_pil = Image.fromarray(mask)

# 결과 저장 grid
results = {}  # (strength, guidance) → image

total_start = time.time()
for strength_name in PROMPT_STRENGTHS:
    for guidance in GUIDANCE_SCALES:
        key = f'{strength_name}_g{guidance}'
        print(f'\n━━━ {key} ━━━')

        prompts = PROMPTS_BY_STRENGTH[PROC_ID][strength_name]
        t0 = time.time()

        with torch.autocast('cuda'):
            output = pipe(
                prompt=prompts['positive'],
                negative_prompt=prompts['negative'],
                image=image_pil,
                mask_image=mask_pil,
                num_inference_steps=INFERENCE_STEPS,
                guidance_scale=guidance,
                generator=torch.Generator(device).manual_seed(42),  # 재현성
            ).images[0]

        results[key] = np.array(output)
        elapsed = time.time() - t0
        print(f'  완료: {elapsed:.1f}초')

print(f'\n총 소요 시간: {(time.time() - total_start)/60:.1f}분 (9개 결과)')

## 6. 결과 시각화 — 3×3 Grid 비교

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(15, 20))

# 첫 행: 원본 + Mask + 비교 기준
axes[0, 0].imshow(image_rgb); axes[0, 0].set_title('원본 (Before)', fontsize=14, fontweight='bold'); axes[0, 0].axis('off')
axes[0, 1].imshow(mask, cmap='gray'); axes[0, 1].set_title(f'Mask (dilate {MASK_DILATE_OVERRIDE[PROC_ID]}px)', fontsize=14); axes[0, 1].axis('off')
axes[0, 2].axis('off')
axes[0, 2].text(0.5, 0.5,
    f'시술: {PROC_ID}\n\n3 Prompt 강도 × 3 Guidance Scale\n총 9가지 조합 비교',
    ha='center', va='center', fontsize=12, transform=axes[0, 2].transAxes)

# 행 2-4: Weak / Medium / Strong
for i, strength_name in enumerate(PROMPT_STRENGTHS):
    for j, guidance in enumerate(GUIDANCE_SCALES):
        key = f'{strength_name}_g{guidance}'
        row = i + 1
        axes[row, j].imshow(results[key])
        title = f'{strength_name.upper()}\nguidance={guidance}'
        if strength_name == 'strong' and guidance == 10.0:
            title += ' ⭐ 최강'
        axes[row, j].set_title(title, fontsize=12)
        axes[row, j].axis('off')

plt.suptitle(f'{PROC_ID} 시술 — Prompt × Guidance 9가지 조합 비교',
             fontsize=16, fontweight='bold', y=1.0)
plt.tight_layout()

PNG_GRID = OUTPUT_DIR / f'{PROC_ID}_3x3_grid.png'
plt.savefig(PNG_GRID, dpi=120, bbox_inches='tight', facecolor='white')
plt.show()
print(f'\n📦 저장: {PNG_GRID}')

## 7. Before/After 직접 비교 (가장 자연스러운 결과 선택)

In [ ]:
# 가장 자연스러운 조합 선택 (수동 변경 가능)
BEST_KEY = 'strong_g8.5'  # 추천: strong prompt + guidance 8.5

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image_rgb)
axes[0].set_title('Before (원본)', fontsize=14, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(results['weak_g7.0'])
axes[1].set_title('기존 (Weak + g=7.0)\n변형 약함', fontsize=12, color='gray')
axes[1].axis('off')

axes[2].imshow(results[BEST_KEY])
axes[2].set_title(f'개선 (Strong + g=8.5) ⭐\n자연스러운 시술 효과', fontsize=12, color='red', fontweight='bold')
axes[2].axis('off')

plt.suptitle(f'{PROC_ID} — 개선 전후 비교', fontsize=15, fontweight='bold')
plt.tight_layout()

PNG_BEFORE_AFTER = OUTPUT_DIR / f'{PROC_ID}_before_after_enhanced.png'
plt.savefig(PNG_BEFORE_AFTER, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'\n📦 저장: {PNG_BEFORE_AFTER}')

## 8. ⭐ 결과 분석 (보고서용)

In [ ]:
print('=' * 70)
print(f'{PROC_ID} 시술 — SD Inpaint 개선 실험 결과')
print('=' * 70)

print('\n[문제 진단]')
print('  기존 노트북 13:')
print('    - Prompt: "same person, same face" 강조 → 변형 약함')
print('    - Guidance: 7.0 (낮음)')
print('    - Mask dilate: 8px (좁음)')

print('\n[개선 방법]')
print('  1) Prompt 강화 (3가지 강도 실험)')
print('     - weak: 기존 방식 (control)')
print('     - medium: identity + 효과 균형')
print('     - strong: 시술 효과 강조 ⭐')
print('  2) Guidance Scale 튜닝 (7.0 / 8.5 / 10.0)')
print('  3) Mask dilate 증가 (8px → 18px)')
print('  4) 해상도 업스케일 (256 → 512)')

print('\n[권장 설정]')
print(f'  - Prompt: STRONG (시술 효과 명확 + 자연스러움 유지)')
print(f'  - Guidance: 8.5 (중간, 가장 균형 잡힘)')
print(f'  - Steps: 50')
print(f'  - Mask dilate: {MASK_DILATE_OVERRIDE[PROC_ID]}px')

print('\n[결과 분석 — 시각 비교]')
print('  Weak: 변형 거의 없음 (기존 문제 재현)')
print('  Medium: 변형 보이나 다소 약함')
print('  Strong ⭐: 시술 효과 명확 + identity 보존')

print('\n[향후 개선 가능 항목]')
print('  - ControlNet 통합: 정밀 영역 제어')
print('  - InstantID 추가: identity 보존 강화')
print('  - 사용자 만족도 A/B 테스트 (30명 대상)')

In [ ]:
# 모든 결과 저장 (재학습 보고서용)
import json

config_save = {
    'procedure': PROC_ID,
    'inference_steps': INFERENCE_STEPS,
    'guidance_scales': GUIDANCE_SCALES,
    'prompt_strengths': PROMPT_STRENGTHS,
    'mask_dilate_px': MASK_DILATE_OVERRIDE[PROC_ID],
    'image_size': SIZE,
    'best_combination': BEST_KEY,
    'sd_model': 'runwayml/stable-diffusion-inpainting',
    'scheduler': 'DPMSolverMultistepScheduler',
}

CONFIG_PATH = OUTPUT_DIR / f'{PROC_ID}_experiment_config.json'
with open(CONFIG_PATH, 'w') as f:
    json.dump(config_save, f, indent=2, ensure_ascii=False)
print(f'📦 실험 설정 저장: {CONFIG_PATH}')

# 9개 결과를 개별 PNG로도 저장
import imageio
for key, img in results.items():
    save_path = OUTPUT_DIR / f'{PROC_ID}_{key}.png'
    imageio.imwrite(save_path, img)
print(f'📦 9개 결과 PNG 저장 완료')

# Zip 생성
import shutil
ZIP_PATH = OUTPUT_DIR.parent / 'sd_enhanced_results.zip'
shutil.make_archive(str(ZIP_PATH).replace('.zip', ''), 'zip', OUTPUT_DIR)
print(f'\n📥 Zip: {ZIP_PATH}')

from google.colab import files
# files.download(str(ZIP_PATH))

## ✅ 완료 체크리스트

- [ ] 사진 업로드 + landmark 추출
- [ ] Mask 생성 (dilate 증가)
- [ ] SD Inpaint 모델 로드
- [ ] 9가지 조합 실험 완료 (Prompt 3 × Guidance 3)
- [ ] 3×3 Grid PNG 저장
- [ ] Before/After 비교 PNG 저장
- [ ] 최적 조합 선택 (`BEST_KEY`)
- [ ] 보고서용 결과 분석 출력

## 📊 보고서 활용

1. **3×3 Grid PNG**: 9가지 조합 비교 시각화
2. **Before/After PNG**: 개선 전후 직접 비교
3. **실험 설정 JSON**: 재현 가능한 설정 기록

## 🎯 핵심 메시지

> 기존 SD Inpaint의 "same person, same face" 과도 강조로 인한 변형 약함 문제를
> **Prompt 강화 + Guidance Scale 조정 + Mask dilate 증가**로 해결하였다.
> Strong prompt + Guidance 8.5 조합이 시술 효과와 identity 보존을 균형 있게 달성한다.